# Tutorial 1 — First Composition

This notebook introduces the core idea of spxa: stochastic processes as first-class algebraic objects.
You write `Z = X + c*Y` or `W = X @ T` and get back a new process with exact analytical properties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spxa
from spxa.zoo.levy import BrownianMotion, VarianceGamma, GammaProcess, NIG
from spxa.analytics import cumulant_table
print('spxa', spxa.__version__)

## 1. Adding two processes

For independent Lévy processes $X$ and $Y$, the sum $Z = X + Y$ has triplet
$$
(b_Z, \sigma^2_Z, \nu_Z) = (b_X + b_Y,\; \sigma^2_X + \sigma^2_Y,\; \nu_X + \nu_Y)
$$
This is **exact** — not an approximation.

In [ ]:
bm  = BrownianMotion(mu=0.0, sigma=0.3)   # pure Gaussian, no jumps
vg  = VarianceGamma(sigma=0.2, nu=0.1, theta=-0.1)  # jump process

Z = bm + vg
print('Z:', Z)
print('Exactness:', Z.exactness.name)
print('Triplet σ²:', Z.triplet.sigma_sq)          # Gaussian part only (VG has σ²=0)
print('Triplet b: ', Z.triplet.b)

## 2. Scalar multiplication

For $Z = c \cdot X$, the triplet transforms as
$$
(b_Z, \sigma^2_Z, \nu_Z) = (cb + \Delta b,\; c^2 \sigma^2,\; \nu(\cdot/c))
$$
The drift correction $\Delta b$ accounts for mass shifting across the unit ball under the truncation function.

In [ ]:
z2 = 2.0 * BrownianMotion(mu=0.0, sigma=1.0)
print('2*BM sigma_sq:', z2.triplet.sigma_sq, '  (expected 4.0)')

## 3. Cumulant table

Every EXACT process exposes analytical cumulants via the Lévy–Khintchine formula.
The cumulant table also computes skewness and excess kurtosis.

In [ ]:
vg2 = VarianceGamma(sigma=0.2, nu=0.1, theta=-0.15)
table = cumulant_table(vg2, order=4, t=1.0)
for k, v in table.items():
    print(f'  {k:<20} {v:.6f}')

## 4. Subordination

Subordinating Brownian motion by a Gamma process gives the Variance Gamma model.
The `@` operator does this: `W = BM @ Gamma`.

In [ ]:
bm0 = BrownianMotion(mu=0.0, sigma=0.2)
g   = GammaProcess(a=10.0, b=10.0)
W   = bm0 @ g   # subordinated process
print('Subordinated process:', W)
print('Exactness:', W.exactness.name)

## 5. The Story: derivation trace

`.__story__()` produces a human-readable derivation of how the current process was constructed.

In [ ]:
Z2 = 0.5 * BrownianMotion(sigma=1.0) + 2.0 * VarianceGamma(sigma=0.2, nu=0.1, theta=0.0)
print(Z2.__story__())

## 6. LaTeX rendering (Jupyter only)

In a Jupyter environment, you can render the derivation as LaTeX:

In [ ]:
# In a Jupyter notebook, use:
# from IPython.display import display, Math
# display(Math(Z2.__story_latex__()))
latex = Z2.__story_latex__()
print(latex[:300], '...')

## 7. Property lattice

Properties propagate automatically. The library is honest when they are lost.

In [ ]:
bm_m  = BrownianMotion(mu=0.0, sigma=1.0)   # martingale
bm_d  = BrownianMotion(mu=1.0, sigma=1.0)   # NOT martingale (drift)
g1    = GammaProcess(a=1.0, b=1.0)            # subordinator

print('BM(0)+BM(0) martingale?    ', (bm_m + bm_m).properties.is_martingale)
print('BM(0)+BM(1) martingale?    ', (bm_m + bm_d).properties.is_martingale)
print('Gamma+Gamma subordinator?  ', (g1 + g1).properties.is_subordinator)
print('(-1)*Gamma subordinator?   ', (-1.0 * g1).properties.is_subordinator)

## 8. Simulation

In [ ]:
rng = np.random.default_rng(42)
vg3 = VarianceGamma(sigma=0.2, nu=0.1, theta=-0.1)
paths = vg3.simulate(n_steps=252, n_paths=5, T=1.0, rng=rng)

fig, ax = plt.subplots(figsize=(9, 4))
t = np.linspace(0, 1, 253)
for i in range(5):
    ax.plot(t, paths[i], lw=0.8)
ax.set_xlabel('Time')
ax.set_ylabel('$X_t$')
ax.set_title('Variance Gamma sample paths')
plt.tight_layout()
plt.savefig('vg_paths.png', dpi=120)
plt.show()
print('paths shape:', paths.shape)